# Discover DB


## Imports e configurações iniciais

In [1]:
import pandas as pd
import os
import requests
import json
from neo4j import GraphDatabase, basic_auth
import wikipedia
import config

## Data loading

Pega dados do crawler

In [33]:
print('Crawler')
def get_crawler(url):
    response = requests.get(url)
    return response.text

response = json.loads(get_crawler(os.getenv('CRAWLER_URL')))

Crawler


## Descobrimento de dados
Descomentar para entender a estrutura

In [34]:
# soma = 0
# for element in response:
#     print(element.keys())
#     print(element['Category'])
#     print(element['Description'])
#     print(element['Link'])
#     print(element['PubDate'])
#     print(element['Title'])
#     print(element['image'])
#     break

## Conectando ao db

In [35]:
driver = GraphDatabase.driver(os.getenv('N4J_URL'),  auth=basic_auth(os.getenv('N4J_USER'),os.getenv('N4J_PASS')))
sess = driver.session()

### Super categorias
Criando nós de super categorias

In [36]:
print('Super categorias')
with driver.session() as sess:
    for element in response:
        if 'Label' in element.keys():
            sess.run("""\
                MERGE (a:AREA {name: $label})
                """, {"label":element['Label']})

Super categorias


### Interesses:
Crio os nós de Interesses, se existir a label (crawler alterado), colcoar a label, se não, apenas criar nó de interesse

#### atualizar depois de trnasicionar db

In [37]:
print('Interesses')
with driver.session() as sess:
    for element in response:
        for interest in element['Category']:
            sess.run("""\
                MERGE (a:INTEREST {name: $name})
                """, {"name":interest})

Interesses


## Artigos 
Primeiro crio os nós de artigos e depois conecto eles a cada nó de categoria desses. E adiciono as features de cada nó.


In [38]:
print('Artigos e interesses')
with driver.session() as sess:
    for element in response:
        sess.run("""\
            MERGE (b:Data:Text:Blog {title: $title})
            SET b.link = $link, b.image_url = $image_url, b.description = $description, b.date = $pub_date
            """, {"link":element['Link'],"image_url":element['image'],"pub_date":element['PubDate'],
                  "description":element['Description'], "title":element['Title']})

Artigos e interesses


### Criando ligações entre nós:

In [39]:
with driver.session() as sess:
    for element in response:
        for interest in element['Category']:
            if 'Label' in element.keys():
                sess.run("""\
                    MATCH (a:INTEREST {name: $name}),(b:Text {title: $title}),(c:AREA {name: $label})
                    MERGE (b)-[r:BELONGS_TO]->(a)
                    MERGE (b)-[:BELONGS_TO]->(c)
                    MERGE (a)<-[:INCLUDES]-(c)
                    """, {"name":interest, "title":element['Title'], "label":element['Label']})
            else:
                sess.run("""\
                    MATCH (a:INTEREST {name: $name}),(b:Text {title: $title})
                    MERGE (b)-[r:BELONGS_TO]->(a)
                    """, {"name":interest, "title":element['Title']})

### Adicionando features aos artigos

### Conectando áreas similares
SE mais de uma categoria aparece no artigo, gerar conexões entre elas. Como tive que iterar por todas, na outra célula apago os `self-loops`

In [41]:
print('Features')
with driver.session() as sess:
    for element in response:
        for interest in element['Category']:
            if len(element['Category']) != 1:    
                sess.run("""\
                    MATCH (a:INTEREST {name: $name}), (b:INTEREST {name: $other}) 
                    WHERE NOT (a)-[:IS_RELATED]-(b) 
                    MERGE (a)-[:IS_RELATED]->(b)
                    """, {"name":element['Category'][0], "other":interest})

Features


In [42]:
### Deletando ligações iguais
with driver.session() as sess:
    for element in response:
        for interest in element['Category']:
            if len(element['Category']) >=1:    
                sess.run("""\
                    MATCH (a:INTEREST {name: $name})-[r:IS_RELATED]->(b:INTEREST {name: $other}) 
                    WHERE a.name = b.name
                    DELETE r
                    """, {"name":element['Category'][0], "other":interest})

### Wikipedia
Uso a api da wikipedia para gerar descrições nos nós de categorias, salvo o arquivo 'wiki.json' para não precisar fazer request denovo, eles são a parte mais demoradade desse código'

In [44]:
print('Wikipedia')
with open('data/wiki.json','r') as fp:
    pre_dict = json.load(fp)

Wikipedia


In [45]:
lista = []
for element in response:
        for interest in element['Category']:
            if interest not in pre_dict.keys():
                lista.append(interest)
interests = set(lista)
descriptions = pre_dict
for element in interests:
    if element not in descriptions.keys():
        try:
            pagina = wikipedia.page(element)
            descriptions[element] = pagina.summary
        except:
            descriptions[element] = 'Not Found'
        

In [46]:
with open('data/wiki.json', 'w') as fp:
    json.dump(descriptions, fp)

In [47]:
with driver.session() as sess:
    for element in descriptions.keys():
        sess.run("""\
                MATCH (b:INTEREST {name: $name})
                SET b.description = $description
                """, {"name":element,"description":descriptions[element]})
    

In [48]:

# for key,value in descriptions.items():
#     if value == 'Not Found':
#         print(key)


networking
alfred
api
dry
aws-deepracer
relationships
portfolio
diversity
backend
vuejs
stem
equality
blackintheivory
race
tips
eightysomethings
humanity
reid
development
codingbootcamp
jobs
spark
inclusion
degendering-ai
tricks
android
apps
developer
usa
equity
mac
linux
lecture-notes
ui
gaming
visualization
improve-reading-skills
nltk
pyspark
agile
delivery
difference
pytest
json
democracy
hive
coding
cluster
advisory
voice-ui
resnet
change
stripe
mobile
invector-labs
tech
blindspots
unity
better-programmer
tokenization
figma
haptics
tracking
identity
optical-character-recogn
ux
rest
gke
games
features
ai
concurrency
etl
pandas-dataframe
intro
ec2-instance
golang-tutorial
terminal
graph
ec
doctors
node
biocomputing
few-shot-learning
beginner
docker-compose
raspberry-pi
csharp
stumpy
razor
feed-forward-networks
advice
filters
digital
resnet-50
function
vector
unions
python3
men
callback
orm
pry
deployment
basics
privilege
bem
vr
colombia
bitcoin
itsm
golang-development
fintech-startup